# Risk Sharing — Day 1: complete working notebook

**Owner:** Nouman, implementation and computation. **Review:** Parsa, model specification and independent hand calculation.

This is a prepared, tested Day 1 starting point, not a completed team sign-off or a solver. The two-agent example below is **provisional** until Parsa confirms or replaces it. Computed outputs included in this file were produced in ChatGPT's execution environment, not your Colab session; rerun every cell in Colab.

## How to use this file
1. Download this `.ipynb` file. In Colab, use **File → Upload notebook** and select it. You can use this notebook instead of retyping code into your blank notebook.
2. Save your own copy in Drive. No GPU is needed for these checks.
3. Run the code cell below from top to bottom (Shift+Enter). It creates a `risk_sharing_day1` project folder and prints checks and welfare values.
4. Confirm the printed checks pass. Read the explanations and compare both allocations with Parsa's independent calculation.
5. Download the generated `risk_sharing_day1_bundle.zip` using Colab's Files panel. This backs up the code, settings, outputs, environment record, and draft model card; runtime files should not be your only copy.
6. After changing agreed inputs, rerun the full cell. It rewrites the generated project files with the current settings.

## The five workdays
| Day | Your work | Evidence required to finish |
|---|---|---|
| 1 | Environment and repository; arrays, ranks, probabilities; two feasible allocations; tiny objective check | Agreed model card; Parsa's independently matching objective values; budget/domain/monotonicity residuals |
| 2 | General objective, gradient, and reverse cumulative sum linear oracle | Directional finite differences and independently formulated HiGHS LP comparison |
| 3 | Conditional-gradient loop, line search, stopping rule, and gap | Feasible small run compared with independent optimizer; welfare F, gap G, and bound F+G |
| 4 | Eight-agent Example 2 | Allocations and welfare compared with Parsa's analytic benchmark on the same grid; table and overlay plot |
| 5 | Grid refinement, tolerance checks, cleanup, rerun instructions | Reproducible baseline package; separate solver and grid errors; joint review of Parsa's memo |

Move on when the checks pass, not just because the calendar advances. Neural-network work comes after the reference solver is trusted.

## Day 1 specification and remaining decisions
The eventual Example 2 target has eight agents, wealth uniform on [0,1], all Pareto weights one, h1 for agents 3–7 and h2 for agents 1,2,8. Each allocation is nonnegative, each row sums to wealth, and each column is nondecreasing.

For today's **provisional two-agent illustration**, use one h1 agent and one h2 agent, weights (1,1), and three midpoint quadrature nodes. These are not the full eight-agent experiment.

- `w`: wealth nodes, shape `(3,)`.
- `r`: probability ranks, shape `(3,)`; separate from wealth even when numerically equal.
- `p`: integration weights/probabilities, shape `(3,)`.
- `C`: consumptions, shape `(3,2)`; rows are states and columns are agents.
- `r = (j + 0.5)/3`, `w = r`, `p = 1/3`. This follows the guide's midpoint convention. It replaces the earlier informal endpoint example `[0, 0.5, 1]`; do not mix those inputs with these outputs.
- `h1(r,c) = log(1 + sqrt(r) + c)`; `h2(r,c) = log(1 + r**(2/3) + c)`. Log means natural logarithm.
- Equal sharing: `(w/2, w/2)`. Other feasible sharing: `(w/3, 2*w/3)`.
- Finite-grid welfare: `F(C) = sum_j p[j] * (h1(r[j], C[j,0]) + h2(r[j], C[j,1]))`.
- Use absolute tolerance `1e-12` for these small floating-point checks. This is a declared software tolerance, not a continuous-model error bound.

Parsa must confirm the tiny model, agent order, weights, ranks, and allocations, or supply replacements. A midpoint sum approximates the continuous rank integral; we are checking that finite sum, not claiming an exact discrete-atom utility calculation.

## Environment and supplied baseline
The guide requests Python 3.12 and reports tested versions Python 3.12.14, NumPy 2.3.5, SciPy 1.18.1, and Matplotlib 3.11.2. This notebook records the versions actually present rather than assuming Colab matches them. It needs NumPy for today's calculation; SciPy and Matplotlib are recorded for later days. Missing NumPy raises an explicit error. Do not silently treat a different environment as an exact reproduction of the supplied environment.

The separate `replication/` code package was not included among the three PDFs. Ask Parsa for it; the notebook records the supplied baseline as **not run**. If supplied, run its documented commands in that folder and retain its logs. The guide's Linux commands are `python -m venv .venv`, `.venv/bin/python -m pip install -r requirements.txt`, `.venv/bin/python run_replication.py --profile verified --out results`, and `.venv/bin/python test_replication.py`. These are for the supplied package, not this independent Day 1 script.

The cell below creates a local Git repository if Git is available; it does not create a GitHub repository or upload anything. The downloadable archive excludes `.git`, so initialize Git again after unpacking if needed.


In [1]:
import sys
import json
import math
import platform
import subprocess
import shutil
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

try:
    import numpy as np
except ImportError as exc:
    raise RuntimeError("NumPy is missing. In Colab run %pip install numpy in a separate cell, then rerun.") from exc

OUT = Path("risk_sharing_day1")
OUT.mkdir(exist_ok=True)
(OUT / "day1_check.py").write_text('import sys\nimport json\nimport math\nimport platform\nimport subprocess\nimport shutil\nfrom pathlib import Path\nfrom importlib.metadata import version, PackageNotFoundError\n\ntry:\n    import numpy as np\nexcept ImportError as exc:\n    raise RuntimeError("NumPy is missing. In Colab run %pip install numpy in a separate cell, then rerun.") from exc\n\nOUT = Path("risk_sharing_day1")\nOUT.mkdir(exist_ok=True)\nTOL = 1e-12\nm, n = 3, 2\nr = (np.arange(m, dtype=float) + 0.5) / m\nw = r.copy()  # Uniform [0,1] ONLY: keep a separate array.\np = np.full(m, 1.0 / m)\nweights = np.ones(n)\nallocations = {\n    "equal": np.column_stack((w / 2, w / 2)),\n    "one_third_two_thirds": np.column_stack((w / 3, 2 * w / 3)),\n}\n\ndef diagnostics(C):\n    C = np.asarray(C, dtype=float)\n    if C.shape != (m, n) or not np.isfinite(C).all():\n        raise ValueError("Allocation must be finite with shape (3,2).")\n    budget = float(np.max(np.abs(C.sum(axis=1) - w)))\n    minimum = float(C.min())\n    increment = float(np.diff(C, axis=0).min())\n    return {\n        "maximum_absolute_budget_residual": budget,\n        "minimum_consumption": minimum,\n        "minimum_monotonicity_increment": increment,\n        "nonnegativity_violation": max(0.0, -minimum),\n        "monotonicity_violation": max(0.0, -increment),\n        "feasible": bool(budget <= TOL and minimum >= -TOL and increment >= -TOL),\n    }\n\ndef welfare(C):\n    utility = np.column_stack((\n        np.log1p(np.sqrt(r) + C[:, 0]),\n        np.log1p(r ** (2.0 / 3.0) + C[:, 1]),\n    ))\n    contributions = p[:, None] * weights[None, :] * utility\n    return float(contributions.sum()), contributions\n\nassert np.isfinite(w).all() and np.isfinite(r).all() and np.isfinite(p).all()\nassert np.all(w >= 0) and np.all(np.diff(w) > 0)\nassert np.all((r > 0) & (r < 1)) and np.all(np.diff(r) > 0)\nassert np.all(p >= 0) and abs(float(p.sum()) - 1.0) <= TOL\nassert weights.shape == (n,) and np.all(weights > 0)\n\nresults = {}\nnp.set_printoptions(precision=10)\nprint("Wealth:", w)\nprint("Ranks:", r)\nprint("Probability sum:", p.sum())\nfor label, C in allocations.items():\n    check = diagnostics(C)\n    assert check["feasible"], (label, check)\n    value, terms = welfare(C)\n    # Scalar arithmetic cross-check, NOT a substitute for Parsa\'s hand calculation.\n    scalar_value = sum(\n        float(p[j]) * (\n            float(weights[0]) * math.log(1 + math.sqrt(float(r[j])) + float(C[j, 0]))\n            + float(weights[1]) * math.log(1 + float(r[j]) ** (2 / 3) + float(C[j, 1]))\n        ) for j in range(m)\n    )\n    assert abs(value - scalar_value) <= TOL\n    results[label] = {**check, "welfare": value,\n                      "scalar_crosscheck_absolute_error": abs(value - scalar_value)}\n    np.savetxt(OUT / (label + ".csv"),\n               np.column_stack((w, r, p, C, terms, terms.sum(axis=1))), delimiter=",",\n               header="wealth,rank,probability,c1,c2,weighted_h1,weighted_h2,state_welfare",\n               comments="", fmt="%.17g")\n    print("\\n" + label)\n    print(C)\n    print("Welfare: %.12f" % value)\n    print(json.dumps(check, indent=2))\n\n# Ensure the feasibility checker actually rejects each kind of violation.\nbad_budget = allocations["equal"].copy()\nbad_budget[1, 0] += 0.01\nassert not diagnostics(bad_budget)["feasible"]\nbad_domain = allocations["equal"].copy()\nbad_domain[0] = [-0.01, w[0] + 0.01]\nassert not diagnostics(bad_domain)["feasible"]\nbad_monotone = allocations["equal"].copy()\nbad_monotone[1] = [0.4, 0.1]\nbad_monotone[2] = [0.3, w[2] - 0.3]\nassert not diagnostics(bad_monotone)["feasible"]\nprint("\\nPASS: input checks, both feasible allocations, scalar objective checks, and three rejection checks.")\n\npackages = {}\nfor name in ["numpy", "scipy", "matplotlib"]:\n    try:\n        packages[name] = version(name)\n    except PackageNotFoundError:\n        packages[name] = "NOT INSTALLED"\nenvironment = {"python": sys.version, "platform": platform.platform(), "packages": packages}\nprint("\\nActual environment:", json.dumps(environment, indent=2))\nif sys.version_info[:2] != (3, 12):\n    print("Recorded deviation: Python differs from the guide\'s requested 3.12.")\nconfig = {"status": "PROVISIONAL: awaiting Parsa", "m": m, "n": n,\n          "wealth": w.tolist(), "ranks": r.tolist(), "probabilities": p.tolist(),\n          "pareto_weights": weights.tolist(), "agent_types": ["h1", "h2"],\n          "quadrature": "uniform midpoint", "tolerance": TOL}\n(OUT / "config.json").write_text(json.dumps(config, indent=2))\n(OUT / "results.json").write_text(json.dumps(results, indent=2))\n(OUT / "environment.json").write_text(json.dumps(environment, indent=2))\n(OUT / "requirements-recorded.txt").write_text("\\n".join(\n    name + "==" + value for name, value in packages.items() if value != "NOT INSTALLED") + "\\n")\n(OUT / "model_card_DRAFT.md").write_text(\n    "# Provisional Day 1 model card\\n"\n    "Owner of final specification: Parsa. Status: awaiting confirmation.\\n"\n    "We allocate nonnegative wealth. Continuous target: uniform [0,1].\\n"\n    "Tiny illustration: 3 midpoint quadrature nodes, 2 agents, types (h1,h2), weights (1,1).\\n"\n    "h1(r,c)=log(1+sqrt(r)+c); h2(r,c)=log(1+r**(2/3)+c).\\n"\n    "F=sum_j p_j sum_i weight_i h_i(r_j,C_ji). Natural logarithms.\\n"\n    "C has shape (states,agents); each row sums to wealth; entries >=0; columns nondecreasing.\\n"\n    "Ranks and wealth are stored separately. See config.json for exact inputs.\\n"\n    "Compare equal sharing and (w/3,2w/3). Neither is claimed optimal.\\n"\n    "Full Example 2 later: 8 agents; h1 for 3-7, h2 for 1,2,8; all weights one.\\n"\n    "Sources: replication guide sections 2-3, printed pages 3-4; five-day plan Day 1, page 2.\\n")\n(OUT / "review_status.json").write_text(json.dumps({\n    "computational_checks": "passed",\n    "Parsa_model_card_confirmation": "pending",\n    "Parsa_independent_objective_comparison": "pending",\n    "supplied_baseline": "not run: separate code package not supplied",\n    "day1_joint_gate": "pending",\n}, indent=2))\n(OUT / "README.md").write_text(\n    "# Day 1 handover\\n"\n    "Provisional example only; joint Day 1 gate is pending Parsa\'s review.\\n"\n    "Rerun the notebook code cell, or execute: python day1_check.py\\n"\n    "Run from the directory containing the risk_sharing_day1 folder.\\n"\n    "The script uses NumPy. requirements-recorded.txt records available project packages;\\n"\n    "environment.json records Python and platform. It is not a full transitive lockfile.\\n"\n    "Outputs are regenerated in risk_sharing_day1; preserve a copy before changing inputs.\\n"\n    "Compare the CSV term tables and objective values with Parsa\'s independent work.\\n"\n    "Do not mark the review passed merely because the script prints PASS.\\n")\nif shutil.which("git"):\n    git_result = subprocess.run(["git", "init", str(OUT)], capture_output=True, text=True)\n    print("Local Git repository:", "initialized" if git_result.returncode == 0 else "not initialized")\nelse:\n    print("Git unavailable: project folder created; repository setup remains pending.")\n# Build a portable archive without Git internals.\nimport zipfile\nwith zipfile.ZipFile("risk_sharing_day1_bundle.zip", "w", zipfile.ZIP_DEFLATED) as archive:\n    for path in sorted(OUT.rglob("*")):\n        if path.is_file() and ".git" not in path.parts:\n            archive.write(path, arcname=str(path))\nprint("\\nSaved project files and risk_sharing_day1_bundle.zip.")\nprint("DAY 1 JOINT SIGN-OFF: PENDING PARSA\'S SPECIFICATION AND INDEPENDENT CALCULATION.")\n')
TOL = 1e-12
m, n = 3, 2
r = (np.arange(m, dtype=float) + 0.5) / m
w = r.copy()  # Uniform [0,1] ONLY: keep a separate array.
p = np.full(m, 1.0 / m)
weights = np.ones(n)
allocations = {
    "equal": np.column_stack((w / 2, w / 2)),
    "one_third_two_thirds": np.column_stack((w / 3, 2 * w / 3)),
}

def diagnostics(C):
    C = np.asarray(C, dtype=float)
    if C.shape != (m, n) or not np.isfinite(C).all():
        raise ValueError("Allocation must be finite with shape (3,2).")
    budget = float(np.max(np.abs(C.sum(axis=1) - w)))
    minimum = float(C.min())
    increment = float(np.diff(C, axis=0).min())
    return {
        "maximum_absolute_budget_residual": budget,
        "minimum_consumption": minimum,
        "minimum_monotonicity_increment": increment,
        "nonnegativity_violation": max(0.0, -minimum),
        "monotonicity_violation": max(0.0, -increment),
        "feasible": bool(budget <= TOL and minimum >= -TOL and increment >= -TOL),
    }

def welfare(C):
    utility = np.column_stack((
        np.log1p(np.sqrt(r) + C[:, 0]),
        np.log1p(r ** (2.0 / 3.0) + C[:, 1]),
    ))
    contributions = p[:, None] * weights[None, :] * utility
    return float(contributions.sum()), contributions

assert np.isfinite(w).all() and np.isfinite(r).all() and np.isfinite(p).all()
assert np.all(w >= 0) and np.all(np.diff(w) > 0)
assert np.all((r > 0) & (r < 1)) and np.all(np.diff(r) > 0)
assert np.all(p >= 0) and abs(float(p.sum()) - 1.0) <= TOL
assert weights.shape == (n,) and np.all(weights > 0)

results = {}
np.set_printoptions(precision=10)
print("Wealth:", w)
print("Ranks:", r)
print("Probability sum:", p.sum())
for label, C in allocations.items():
    check = diagnostics(C)
    assert check["feasible"], (label, check)
    value, terms = welfare(C)
    # Scalar arithmetic cross-check, NOT a substitute for Parsa's hand calculation.
    scalar_value = sum(
        float(p[j]) * (
            float(weights[0]) * math.log(1 + math.sqrt(float(r[j])) + float(C[j, 0]))
            + float(weights[1]) * math.log(1 + float(r[j]) ** (2 / 3) + float(C[j, 1]))
        ) for j in range(m)
    )
    assert abs(value - scalar_value) <= TOL
    results[label] = {**check, "welfare": value,
                      "scalar_crosscheck_absolute_error": abs(value - scalar_value)}
    np.savetxt(OUT / (label + ".csv"),
               np.column_stack((w, r, p, C, terms, terms.sum(axis=1))), delimiter=",",
               header="wealth,rank,probability,c1,c2,weighted_h1,weighted_h2,state_welfare",
               comments="", fmt="%.17g")
    print("\n" + label)
    print(C)
    print("Welfare: %.12f" % value)
    print(json.dumps(check, indent=2))

# Ensure the feasibility checker actually rejects each kind of violation.
bad_budget = allocations["equal"].copy()
bad_budget[1, 0] += 0.01
assert not diagnostics(bad_budget)["feasible"]
bad_domain = allocations["equal"].copy()
bad_domain[0] = [-0.01, w[0] + 0.01]
assert not diagnostics(bad_domain)["feasible"]
bad_monotone = allocations["equal"].copy()
bad_monotone[1] = [0.4, 0.1]
bad_monotone[2] = [0.3, w[2] - 0.3]
assert not diagnostics(bad_monotone)["feasible"]
print("\nPASS: input checks, both feasible allocations, scalar objective checks, and three rejection checks.")

packages = {}
for name in ["numpy", "scipy", "matplotlib"]:
    try:
        packages[name] = version(name)
    except PackageNotFoundError:
        packages[name] = "NOT INSTALLED"
environment = {"python": sys.version, "platform": platform.platform(), "packages": packages}
print("\nActual environment:", json.dumps(environment, indent=2))
if sys.version_info[:2] != (3, 12):
    print("Recorded deviation: Python differs from the guide's requested 3.12.")
config = {"status": "PROVISIONAL: awaiting Parsa", "m": m, "n": n,
          "wealth": w.tolist(), "ranks": r.tolist(), "probabilities": p.tolist(),
          "pareto_weights": weights.tolist(), "agent_types": ["h1", "h2"],
          "quadrature": "uniform midpoint", "tolerance": TOL}
(OUT / "config.json").write_text(json.dumps(config, indent=2))
(OUT / "results.json").write_text(json.dumps(results, indent=2))
(OUT / "environment.json").write_text(json.dumps(environment, indent=2))
(OUT / "requirements-recorded.txt").write_text("\n".join(
    name + "==" + value for name, value in packages.items() if value != "NOT INSTALLED") + "\n")
(OUT / "model_card_DRAFT.md").write_text(
    "# Provisional Day 1 model card\n"
    "Owner of final specification: Parsa. Status: awaiting confirmation.\n"
    "We allocate nonnegative wealth. Continuous target: uniform [0,1].\n"
    "Tiny illustration: 3 midpoint quadrature nodes, 2 agents, types (h1,h2), weights (1,1).\n"
    "h1(r,c)=log(1+sqrt(r)+c); h2(r,c)=log(1+r**(2/3)+c).\n"
    "F=sum_j p_j sum_i weight_i h_i(r_j,C_ji). Natural logarithms.\n"
    "C has shape (states,agents); each row sums to wealth; entries >=0; columns nondecreasing.\n"
    "Ranks and wealth are stored separately. See config.json for exact inputs.\n"
    "Compare equal sharing and (w/3,2w/3). Neither is claimed optimal.\n"
    "Full Example 2 later: 8 agents; h1 for 3-7, h2 for 1,2,8; all weights one.\n"
    "Sources: replication guide sections 2-3, printed pages 3-4; five-day plan Day 1, page 2.\n")
(OUT / "review_status.json").write_text(json.dumps({
    "computational_checks": "passed",
    "Parsa_model_card_confirmation": "pending",
    "Parsa_independent_objective_comparison": "pending",
    "supplied_baseline": "not run: separate code package not supplied",
    "day1_joint_gate": "pending",
}, indent=2))
(OUT / "README.md").write_text(
    "# Day 1 handover\n"
    "Provisional example only; joint Day 1 gate is pending Parsa's review.\n"
    "Rerun the notebook code cell, or execute: python day1_check.py\n"
    "Run from the directory containing the risk_sharing_day1 folder.\n"
    "The script uses NumPy. requirements-recorded.txt records available project packages;\n"
    "environment.json records Python and platform. It is not a full transitive lockfile.\n"
    "Outputs are regenerated in risk_sharing_day1; preserve a copy before changing inputs.\n"
    "Compare the CSV term tables and objective values with Parsa's independent work.\n"
    "Do not mark the review passed merely because the script prints PASS.\n")
if shutil.which("git"):
    git_result = subprocess.run(["git", "init", str(OUT)], capture_output=True, text=True)
    print("Local Git repository:", "initialized" if git_result.returncode == 0 else "not initialized")
else:
    print("Git unavailable: project folder created; repository setup remains pending.")
# Build a portable archive without Git internals.
import zipfile
with zipfile.ZipFile("risk_sharing_day1_bundle.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and ".git" not in path.parts:
            archive.write(path, arcname=str(path))
print("\nSaved project files and risk_sharing_day1_bundle.zip.")
print("DAY 1 JOINT SIGN-OFF: PENDING PARSA'S SPECIFICATION AND INDEPENDENT CALCULATION.")


Wealth: [0.1666666667 0.5          0.8333333333]
Ranks: [0.1666666667 0.5          0.8333333333]
Probability sum: 1.0

equal
[[0.0833333333 0.0833333333]
 [0.25         0.25        ]
 [0.4166666667 0.4166666667]]
Welfare: 1.236217807544
{
  "maximum_absolute_budget_residual": 0.0,
  "minimum_consumption": 0.08333333333333333,
  "minimum_monotonicity_increment": 0.16666666666666669,
  "nonnegativity_violation": 0.0,
  "monotonicity_violation": 0.0,
  "feasible": true
}

one_third_two_thirds
[[0.0555555556 0.1111111111]
 [0.1666666667 0.3333333333]
 [0.2777777778 0.5555555556]]
Welfare: 1.235554019370
{
  "maximum_absolute_budget_residual": 0.0,
  "minimum_consumption": 0.05555555555555555,
  "minimum_monotonicity_increment": 0.1111111111111111,
  "nonnegativity_violation": 0.0,
  "monotonicity_violation": 0.0,
  "feasible": true
}

PASS: input checks, both feasible allocations, scalar objective checks, and three rejection checks.

Actual environment: {
  "python": "3.12.14 (main, Aug 25

## Expected results and how to interpret them

| Allocation | Welfare on this three-node grid |
|---|---:|
| Equal sharing | 1.236217807544 |
| One third / two thirds | 1.235554019370 |

Both should be feasible. Probability sum should be 1 within 1e-12. Maximum budget error should be at most 1e-12. Minimum consumption and minimum monotonicity increment should be nonnegative, allowing 1e-12 rounding tolerance. A larger welfare value between these two candidates does not prove global optimality.

For hand calculation, use r=w=(1/6,1/2,5/6). Equal-sharing rows are (1/12,1/12), (1/4,1/4), (5/12,5/12). Other-sharing rows are (1/18,1/9), (1/6,1/3), (5/18,5/9).

For each row j, calculate `A_j = log(1 + sqrt(r_j) + c1_j)` and `B_j = log(1 + r_j**(2/3) + c2_j)`. Then calculate `(A_1+B_1+A_2+B_2+A_3+B_3)/3`. This is the independent calculator exercise to request from Parsa, using the confirmed specification. The CSVs include every weighted term to locate disagreements.

Our scalar and vectorized implementations agree within tolerance, but both were prepared here. **They do not satisfy the requirement for Parsa's independent calculation.** If hand values are rounded, agree a suitable comparison tolerance based on retained digits; do not demand 1e-12 from a six-decimal calculator answer.

## What to send to Parsa at review
- Draft model card and exact inputs in `config.json`.
- Both allocation CSVs and objective/feasibility values in `results.json`.
- Recorded environment and the missing supplied-code-package status.
- Ask him to confirm or replace the illustrative specification, and compare his two independently computed welfare values.
- Record disagreements and resolve them before moving to Day 2.

## Day 1 is finished only when
1. The project environment and repository are set up and commands/versions recorded.
2. The supplied baseline has been run if its code is available; otherwise its unavailability is recorded and raised with Parsa.
3. Parsa's model card and tiny example are saved and agreed.
4. Probabilities, shapes, domain, statewise budget, and monotonicity checks pass for equal sharing and the other allocation.
5. Your objective values match Parsa's independent calculations to an agreed tolerance.
6. The notebook and results are saved outside the temporary Colab runtime and reviewed together.

If steps 3 or 5 are pending, report **implementation ready for review; Day 1 joint gate pending**, not “Day 1 complete.”

## Troubleshooting
- `NameError`: rerun the entire code cell; do not run fragments out of order.
- Missing NumPy: run `%pip install numpy` in a separate Colab cell, then rerun this cell.
- An assertion fails: retain the exact error and inputs. Do not loosen tolerances until the cause is understood.
- Results differ: check midpoint versus endpoint grid, natural logarithms, agent order, weights, and probability factors first.
- After a runtime reset: rerun the notebook to regenerate outputs; use your saved notebook and downloaded archive.

## Sources
- `Parsa_and_Nouman_Five_Day_Research_Plan (1).pdf`, Day 1 on page 2; daily roadmap and later gates on pages 1–4.
- `replication_guide.pdf`, environment on printed page 3; wealth, utilities and midpoint convention on page 4.

No supplied baseline or full eight-agent optimization was executed in this Day 1 notebook.
